#Title: GLiNER2 for Performing Multiple NLP Tasks Efficiently

#### Member Names :

Lok Prakash Pandey, Samuel Ukoha, Xinquan Yu

#### Email :

[ lok.pandey@torontomu.ca, samuel.ukoha@toronmu.ca, xinquan.yu@torontomu.ca ]

# Introduction

#### Problem Description:
NLP tasks such as Named Entity Recognition (NER), Text Classification (TC), Structured Data Extraction (SDE) and Relation Extraction (RE) requires separate LLM models for each task. This increases engineering complexity and cost of the tasks associated.

#### Context of the Problem:
Performing NLP tasks is important in the real-world because they help with various tasks like understanding documents, classification of documents, data analysis and creating knowledge graphs. A practical NLP solution should be efficient, flexible, and easy to deploy on a limited hardware.

#### Limitations of Other Approaches:
- Traditional LLMs do exist to solve the problems of NER, TC, SDE and RE, but they require expensive GPUs [1].
- Task-based models, which means requiring task specific models to work on different NLP tasks, adds complexity to the NLP process which can be decreased by having one generic model [2].

#### Solution:
This project studies **GLiNER2** (Generalist and Lightweight model for Named Entity Recognition 2), which is a unified system for performing NER, TC, SDE and RE tasks. The implementation provided below reproduces its main usage pattern in Google Colab using the open-source package and pretrained model.

#Background

The following table summarizes the works done relevant to this project:

<table>
<tr>
<th>Reference</th>
<th>Explanation</th>
<th>Dataset/Input</th>
<th>Weakness</th>
</tr>

<tr>
<td>Zaratiana et al. [1]</td>
<td>
They proposed <b>GLiNER</b>, a generalist NER model using a bidirectional transformer.<br><br>
It leverages label descriptions to recognize both seen and unseen entity types without retraining.
</td>
<td>
Standard NER datasets with:<br>
• Text input<br>
• Label descriptions
</td>
<td>
Limited to NER tasks only
</td>
</tr>

<tr>
<td>Zaratiana et al. [2]</td>
<td>
They proposed <b>GLiNER2</b>, an extension of GLiNER.<br><br>
It supports multi-task information extraction using a schema-driven interface for entities and relations.
</td>
<td>
Multi-task IE datasets with:<br>
• Schema definitions (entities, relations)<br>
• Text input
</td>
<td>
Depends on schema design quality <br> and requires careful schema definition
</td>
</tr>

</table>

# Methodology

In this project, we implemented a simplified schema-driven information extraction method based on GLiNER2. We used the released pretrained model in Google Colab rather than training the model from scratch. The main idea of the method is that a single encoder-based model can perform different NLP tasks when given an appropriate schema, such as entity labels, classification labels, relation types, or structured fields.

Our implementation followed one unified workflow. First, we loaded the pretrained GLiNER2 model. Next, we defined schemas depending on the information we wanted to extract. Then, we ran the same model on input text and examined the outputs. Using this process, we tested the framework on classification, named entity recognition, relation extraction, and structured extraction examples. Although the outputs were different, they all came from the same schema-driven extraction method. This allowed us to study the central contribution of the paper in a manageable and practical way.

In [ ]:
!rm -rf /content/gliner2-base-v1

In [ ]:
!pip install -q gliner2
!apt-get -qq update
!apt-get -qq install git-lfs
!git lfs install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 88.5 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Git LFS initialized.


In [ ]:
!git clone https://huggingface.co/fastino/gliner2-base-v1 /content/gliner2-base-v1

Cloning into '/content/gliner2-base-v1'...
remote: Enumerating objects: 38, done.
remote: Total 38 (delta 0), reused 0 (delta 0), pack-reused 38 (from 1)
Receiving objects: 100% (38/38), 1.88 MiB | 6.58 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [ ]:
!ls -lh /content/gliner2-base-v1

total 806M
-rw-r--r-- 1 root root  230 Apr 14 02:31 added_tokens.json
-rw-r--r-- 1 root root  236 Apr 14 02:31 config.json
drwxr-xr-x 2 root root 4.0K Apr 14 02:31 encoder_config
-rw-r--r-- 1 root root 796M Apr 14 02:32 model.safetensors
-rw-r--r-- 1 root root 4.7K Apr 14 02:31 README.md
-rw-r--r-- 1 root root 2.4K Apr 14 02:31 special_tokens_map.json
-rw-r--r-- 1 root root 2.4M Apr 14 02:31 spm.model
-rw-r--r-- 1 root root 3.2K Apr 14 02:31 tokenizer_config.json
-rw-r--r-- 1 root root 8.3M Apr 14 02:31 tokenizer.json


In [ ]:
from gliner2 import GLiNER2

# This line loads the pretrained GLiNER2 model from the local folder.
# "extractor" is the main object we will use for all later tasks.
# We use the same model for classification, NER, relation extraction,
# and structured extraction, which matches the paper's unified framework idea.
extractor = GLiNER2.from_pretrained("/content/gliner2-base-v1")

# This print statement is only used to confirm that the model was loaded successfully.
print("Model loaded successfully.")

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Model loaded successfully.


# 1. Text Classification

In [ ]:
# This section demonstrates text classification.
# The goal is to let the model assign one sentiment label
# to the input sentence.

# "schema" defines the task we want the model to perform.
# Here, we tell GLiNER2 that the task name is "sentiment".
# The candidate labels are "positive", "negative", and "neutral".
# Because we do not set multi_label=True, the model will choose only one label.
schema = extractor.create_schema().classification(
    "sentiment",
    ["positive", "negative", "neutral"]
)

# "text" is the input sentence that we want to classify.
# This sentence expresses a strong positive opinion.
text = "This product exceeded my expectations! Absolutely love it."

# extractor.extract(...) applies the schema to the input text.
# Since the schema is a classification schema, the model will return
# one sentiment label for this sentence.
results = extractor.extract(text, schema)

# Print the final output.
# We expect something similar to: {'sentiment': 'positive'}
print(results)

{'sentiment': 'positive'}


# 2. Named Entity Recognition (NER)

In [ ]:
# This section demonstrates named entity recognition (NER).
# The goal is to identify important entities in the sentence,
# such as company names, people, products, locations, and dates.

# This schema tells the model which entity types to extract.
# The model will scan the text and try to find spans that belong
# to these five categories.
schema = extractor.create_schema().entities([
    "company", "person", "product", "location", "date"
])

# This is the input sentence for NER.
# It contains multiple entity types:
# - Apple Inc. -> company
# - Tim Cook -> person
# - iPhone 15 -> product
# - Cupertino, California -> location
# - September 12, 2023 -> date
text = "Apple Inc. CEO Tim Cook announced the new iPhone 15 in Cupertino, California on September 12, 2023."

# extractor.extract(...) now performs entity extraction because the schema
# is defined with .entities(...).
# The output should group the extracted entities by their type.
results = extractor.extract(text, schema)

# Print the extracted entities.
# The output should be a dictionary with an "entities" field.
print(results)

{'entities': {'company': ['Apple Inc.'], 'person': ['Tim Cook'], 'product': ['iPhone 15'], 'location': ['California', 'Cupertino'], 'date': ['September 12, 2023']}}


# 3. Relation Extraction

In [ ]:
# This section demonstrates relation extraction.
# The goal is not only to find entities, but also to identify
# the relationship between them.

# Here, the schema defines two relation types:
# - "works_for"
# - "lives_in"
# The model will try to detect these semantic relations in the text.
schema = extractor.create_schema().relations([
    "works_for", "lives_in"
])

# This input sentence contains two relations:
# - John works for Apple Inc.
# - John lives in San Francisco.
text = "John works for Apple Inc. and lives in San Francisco."

# extractor.extract(...) performs relation extraction because the schema
# is defined with .relations(...).
# The output should return relation pairs in the form:
# (source entity, target entity)
results = extractor.extract(text, schema)

# Print the extracted relations.
# We expect something similar to:
# {
#   'relation_extraction': {
#       'works_for': [('John', 'Apple Inc.')],
#       'lives_in': [('John', 'San Francisco')]
#   }
# }
print(results)

{'relation_extraction': {'works_for': [('John', 'Apple Inc.')], 'lives_in': [('John', 'San Francisco')]}}


# 4. Structured / JSON Extraction

In [ ]:
# This section demonstrates structured extraction.
# Unlike simple classification or entity extraction,
# the goal here is to return a structured JSON-like output.

# This is the input sentence.
# It contains product-related information:
# - product name
# - price
# - features
text = "The MacBook Pro costs $1999 and features M3 chip, 16GB RAM, and 512GB storage."

# extractor.extract_json(...) is used for structure-only extraction.
# We define one structure called "product".
# Inside "product", we ask the model to extract:
# - "name::str"     -> a single string value for the product name
# - "price"         -> price field (default behavior is allowed)
# - "features"      -> multiple feature values
#
# In other words, we are telling the model to organize the information
# into a JSON-like object instead of only returning labels.
results = extractor.extract_json(
    text,
    {
        "product": [
            "name::str",
            "price",
            "features"
        ]
    }
)

# Print the structured output.
# We expect something like:
# {
#   'product': [{
#       'name': 'MacBook Pro',
#       'price': ['$1999'],
#       'features': ['M3 chip', '16GB RAM', '512GB storage']
#   }]
# }
print(results)

{'product': [{'name': 'MacBook Pro', 'price': ['$1999'], 'features': ['M3 chip', '16GB RAM', '512GB storage']}]}


# Conclusion and Future Direction

This project helped us understand how a single pretrained model can be used as a unified information extraction framework. Our implementation showed that GLiNER2 can support several NLP tasks through schema design, which makes it more flexible than systems built for only one task. It was also practical for our project because we could run the released model in Colab without reproducing a large training pipeline.

However, our work also had limitations. We implemented the released framework rather than training the full method from scratch, so our project should be seen as a simplified implementation. We also found that the output quality can depend on how clearly the schema and labels are defined. In future work, this method could be evaluated more systematically on benchmark datasets and compared more carefully against specialized task-specific models. It would also be useful to explore hierarchical structured extraction further, since the paper notes that this part still lacks strong zero-shot benchmark evaluation.

#References

[1]: Urchade Zaratiana, Nadi Tomeh, Pierre Holat, Thierry Charnois, GLiNER: Generalist Model for Named Entity Recognition using Bidirectional Transformer, arXiv, 2023

[2]: Urchade Zaratiana, Gil Pasternak, Oliver Boyd, George Hurn-Maloney, Ash Lewis, GLiNER2: An Efficient Multi-Task Information Extraction System with Schema-Driven Interface, Proceedings of the 2025 Conference on Empirical Methods in Natural Language Processing: System Demonstrations, 2025, pages 130–140